# List Recoloring of Sparse Graphs

In [1]:
import networkx as nx
import random  # to generate random lists
import math

### Naive List Implementation (yielding $O(n^2)$ running time)

In [2]:
def extend_rseq(coloring: dict, recolor_seq: list, v: int, v_colors: set, 
                   v_source: int, v_target: int, v_nbrs: set, DEBUG=False) -> tuple[dict, list]:
    def dprint(a):
        if DEBUG:
            print(a)
            
    #coloring = coloring0.copy()
    #recolor_seq = recolor_seq0.copy()
    coloring[v] = v_source
    dprint(f"v_nbrs is {v_nbrs}")
    nbrs_r_seq = [(num, vert, color) for num, (vert, color) in enumerate(recolor_seq) if vert in v_nbrs]
    dprint(nbrs_r_seq)
    curr_color = {v:coloring[v] for v in (v_nbrs | {v}) }  # dictionary that maintains colors on N[v]
    how_many = len(v_colors) - len(v_nbrs) - 1

    ext_to_v_seq = []
    for i in range(len(nbrs_r_seq)):
        num, vert, color = nbrs_r_seq[i]
        dprint(f"nbrs_r_seq[i] is {nbrs_r_seq[i]}")
        if color != curr_color[v]:
            curr_color[vert] = color
        else:
            to_avoid1 = {curr_color[w] for w in v_nbrs}
            dprint(f"to_avoid1 is {to_avoid1}.")
            to_avoid2 = {nbrs_r_seq[i+j][2] for j in range(min(how_many,len(nbrs_r_seq)-i))}
            dprint(f"to_avoid2 is {to_avoid2}.")
            allowed_colors = v_colors - (to_avoid1 | to_avoid2)
            dprint(f"allowed_colors is {allowed_colors}.")
            picked_color = next(iter(allowed_colors))
            dprint(f"picked_color is {picked_color}.")
            curr_color[v] = picked_color
            curr_color[vert] = color
            ext_to_v_seq.append((num, picked_color))

    # ext_to_v_seq ... (num, color)  # list: each step recolors v with color, just before step num in recolor_seq 
    #print(f"ext_to_v_seq is {ext_to_v_seq}")
    if (len(ext_to_v_seq) == 0 and v_target != v_source) or (len(ext_to_v_seq) >= 1 and ext_to_v_seq[-1][1] != v_target):
        ext_to_v_seq.append((len(recolor_seq), v_target))
    dprint(f"ext_to_v_seq is {ext_to_v_seq}")
    
    for i in range(len(ext_to_v_seq)-1,-1,-1):  # insert in order of decreasing index to keep indexing consistent
        dprint(f"recolor_seq is {recolor_seq}")
        dprint(f"i is {i}")
        recolor_seq.insert(ext_to_v_seq[i][0], (v, ext_to_v_seq[i][1]))
        dprint(f"recolor_seq is {recolor_seq}")
    return coloring, recolor_seq

In [3]:
def verify_rseq(G, rseq: list, start_coloring: dict, end_coloring: dict, lists: dict, info=False):
    assert all(start_coloring[v] != start_coloring[w] for v,w in G.edges), "Starting coloring is improper."
    assert all(end_coloring[v] != end_coloring[w] for v,w in G.edges), "Ending coloring is improper."
    curr_coloring = start_coloring.copy()
    for num, (v, color) in enumerate(rseq, start=1):
        assert color in lists[v], f"Cannot recolor {v} to {color} at step {num}, since {color} is not in its list: {lists[v]}."
    for num, (v, color) in enumerate(rseq, start=1):
        curr_coloring[v] = color
        for w in G.neighbors(v):
            assert curr_coloring[v] != curr_coloring[w], f"At step {num} cannot recolor {v} with {color}, since it is already used on {w}."
    print("Recoloring sequence is valid.")
    if info:
        counts = {v:0 for v in G}
        for (v, color) in rseq:
            counts[v] += 1
        assert len(rseq) == sum(counts[v] for v in G)
        max_times = max(counts.values())
        min_times = min(counts.values())
        summary_info = {"max_times": max_times, "min_times": min_times, "total_times": len(rseq)}
        return summary_info
    

### New Example on $P_9$

In [4]:
path9_coloring1 = {i:(i%3)+1 for i in range(9)}
path9_coloring2 = {i:3- i%3 for i in range(9)}
print(f"Coloring 1: {path9_coloring1} and Coloring 2: {path9_coloring2}")

Coloring 1: {0: 1, 1: 2, 2: 3, 3: 1, 4: 2, 5: 3, 6: 1, 7: 2, 8: 3} and Coloring 2: {0: 3, 1: 2, 2: 1, 3: 3, 4: 2, 5: 1, 6: 3, 7: 2, 8: 1}


### Linked Lists with Event Dictionaries

To speed up the implementation, we will store everything in a treap.  This is a binary search tree (with events stored at internal nodes) where each node has a priority and each parent has priority at least as large as the priorities of each of its children. 
The recoloring sequence is stored implicitly in the treap and can be recovered by an in-order traversal.  The main point is that if the graph is $d$-degenerate, then we can insert a new vertex in (expected) time $d^2\log n$.  We do this $n$ times, and then at the end, we do an in-order traversal to extract the recoloring sequence.  (We don't need to have a current copy of the recoloring sequence after every insertion.)  For each vertex v, we will store pointers to the nodes of the treap when it gets updated.  The key points are as follows:

1. the expected depth of the treap will be $O(\log n)$
2. for each recoloring event, we can compute its "rank" (position in the recoloring sequence) in time proportional to its depth in the treap
3. for each vertex already inserted (into the graph), we store a list of its recoloring events
4. when we insert a new vertex $v$, we collect all of the recoloring events of its neighbors, compute the rank for each event, and sort these events into a list (ordered by their ranks).  Then we determine any needed recoloring events for v, create a list for these (stored with v), and insert each of these events into the treap
5. The expensive parts of this process are (a) calculating the rank for each neighbor event $O(d(d+1)\log n)$ and (b) rebalancing the treap after we insert a recoloring event for $v$: $O(\log n)$ for one, so $O((d+1)\log n)$ in total

We start with a simple helper function to confirm that a recoloring sequence that we've constructed is valid.  By default, it just returns "Recoloring sequence is valid." or raises an assertion error, if it's not.  But we can also get a bit more information with the option ```info=True```.

In [5]:
def verify_rseq(G, rseq: list, start_coloring: dict, end_coloring: dict, lists: dict, info=False):
    # most of the inputs should be clear.  But lists is a dictionary with the list (set) of allowable colors for each vertex.
    assert all(start_coloring[v] != start_coloring[w] for v,w in G.edges), "Starting coloring is improper."
    assert all(end_coloring[v] != end_coloring[w] for v,w in G.edges), "Ending coloring is improper."
    curr_coloring = start_coloring.copy()
    for num, (v, color) in enumerate(rseq, start=1):
        assert color in lists[v], f"Cannot recolor {v} to {color} at step {num}, since {color} is not in its list: {lists[v]}."
    for num, (v, color) in enumerate(rseq, start=1):
        curr_coloring[v] = color
        for w in G.neighbors(v):
            assert curr_coloring[v] != curr_coloring[w], f"At step {num} cannot recolor {v} with {color}, since it is already used on {w}."
    print("Recoloring sequence is valid.")
    if info:
        counts = {v:0 for v in G}
        for (v, color) in rseq:
            counts[v] += 1
        assert len(rseq) == sum(counts[v] for v in G)
        max_times = max(counts.values())
        min_times = min(counts.values())
        summary_info = {"max_times": max_times, "min_times": min_times, "total_times": len(rseq)}
        return summary_info

The main work of this section is implementing the ```Treap``` class.  A version of treaps is already implemented for Python, but it does not (directly) offer the functionality we need here.  Specifically, we need to be able to maintain a pointer to an event (node in the treap) and insert a new event immediately before that event, even if that event has moved around in the treap since we inserted it.  (Since each node of the treap will be an event, we also implement the ```Event``` class.)

In [6]:
class Event:
    def __init__(self, v, color, priority):
        self.size = 1
        self.priority = priority
        self.vertex = v
        self.color = color

        self.lchild = None
        self.rchild = None
        self.parent = None

    def find_rank(self):
        curr_rank = 1
        curr_spot = self
        if self.lchild is not None:
            curr_rank += self.lchild.size
        while curr_spot.parent is not None:
            if curr_spot.parent.rchild == curr_spot:  # curr_spot is rchild of its parent
                curr_spot = curr_spot.parent
                curr_rank += 1
                if curr_spot.lchild is not None:
                    curr_rank += curr_spot.lchild.size
            else:
                curr_spot = curr_spot.parent
        
        return curr_rank

    # For in in-order traversal, we may want to just output for each event its vertex and color (which we do by default)
    # but we may also want to print the priority and/or the rank
    def print_event(self, priority=False, rank=False):
        if priority:
            if rank:
                print(f"((({self.vertex},{self.color}), priority = {self.priority}), rank = {self.find_rank()})")
            else:
                print(f"(({self.vertex},{self.color}), priority = {self.priority})")
        else:
            if rank:
                print(f"(({self.vertex},{self.color}), rank = {self.find_rank()})")
            else:
                print(f"({self.vertex},{self.color})")

# a minor helper function, so that we can handle the cases lchild = None and rchild = None in a unified way
def size(node):
    if node is None:
        return 0
    else:
        return node.size

# This class is where the bulk of the work happens.
class Treap:
    def __init__(self):
        self.root = None

    # this is how we can print the current (implicit) recoloring sequence at any point, possibly along with priorities and/or ranks
    def inorder_traversal(self, priority=False, rank=False):
    
        def inorder_rec(event):
            if event.lchild is not None:
                inorder_rec(event.lchild)
            event.print_event(priority = priority, rank = rank)
            if event.rchild is not None:
                inorder_rec(event.rchild)
    
        if self.root is not None:
            inorder_rec(self.root)

    # this is how we extract the recoloring sequence at the end
    # similar to inorder_traversal, except that we print nothing and return the whole recoloring sequence as a list
    def get_list(self, priority=False, rank=False):
        rseq_as_list = []
    
        def get_list_rec(event):
            nonlocal rseq_as_list
            if event.lchild is not None:
                get_list_rec(event.lchild)
            rseq_as_list.append((event.vertex, event.color))
            if event.rchild is not None:
                get_list_rec(event.rchild)
    
        if self.root is not None:
            get_list_rec(self.root)
        return rseq_as_list

    # each time that we recolor a vertex we must create an event for that recoloring, and insert that event into the treap
    def insert_event(self, event, next_event):
        # first we handle the case where the treap is empty, i.e., this is the first insertion
        if self.root is None and next_event is None:
            self.root = event
            return
        # next we handle the case where we are "appending" to the treap
        # this is a bit different than normal, because we usual give the pointer to what we insert "right before"
        elif next_event is None:
            curr = self.root
            while curr.rchild is not None:
                curr.size += 1
                curr = curr.rchild
            curr.size += 1
            curr.rchild = event
            event.parent = curr

            # if we appended, then now we need to restore the heap invariant (make priorities satisfy the required inequality)
            curr = event
            while (curr.parent is not None) and (curr.priority > curr.parent.priority):
                self.rotate_up_left(curr)
            return
            
        # if we reach here (and do not return above), then we are in a "normal" insertion case
        # either we insert as a left child of next_event (if no left child exists)
        if next_event.lchild is None:
            next_event.lchild = event
            event.parent = next_event
        # or we insert as the rightmost descendant of a left child (of next_event) if that left child exists
        else:
            curr_event = next_event.lchild
            while curr_event.rchild is not None:
                curr_event = curr_event.rchild
            curr_event.rchild = event
            event.parent = curr_event

        # now we must increase size by 1 for each ancestor on a path to the root
        path_to_root = event
        while path_to_root.parent is not None:
            path_to_root = path_to_root.parent
            path_to_root.size += 1

        # and finally, we restore the invariant for priorities  (ensure that it is still a heap)
        while (event.parent is not None) and (event.priority > event.parent.priority):
            if event.parent.rchild == event:  # event is right child of parent
                self.rotate_up_left(event)
            else:  # event is left child of parent
                self.rotate_up_right(event)
            

    # the way that we main the heap invariant is by rotating events with higher priorities up toward the root
    # either we rotate_up_right or we rotate_up_left
    def rotate_up_right(self, event):
        # assumes parent.lchild == event
        parent_parent = event.parent.parent
        parent = event.parent
        root_to_shift = event.rchild

        parent.lchild = root_to_shift

        if root_to_shift is not None:
            root_to_shift.parent = parent
        parent.parent = event
        event.rchild = parent
     
        event.parent = parent_parent
        if parent_parent is None:
            self.root = event
        else:
            if parent_parent.rchild == parent:
                parent_parent.rchild = event
            else:  # parent_parent.lchild == parent:
                parent_parent.lchild = event
        parent.size = 1 + size(parent.lchild) + size(parent.rchild)
        event.size = 1 + size(event.lchild) + size(event.rchild)

    def rotate_up_left(self, event):
        # assumes parent.rchild == event
        parent_parent = event.parent.parent
        parent = event.parent
        root_to_shift = event.lchild

        parent.rchild = root_to_shift

        if root_to_shift is not None:
            root_to_shift.parent = parent
        parent.parent = event 
        event.lchild = parent
        
        event.parent = parent_parent
        if parent_parent is None:
            self.root = event
        else:
            if parent_parent.rchild == parent:
                parent_parent.rchild = event
            else:  # parent_parent.lchild == parent:
                parent_parent.lchild = event
        parent.size = 1 + size(parent.lchild) + size(parent.rchild)
        event.size = 1 + size(event.lchild) + size(event.rchild)

    def check_invariants(self):  # Mainly just for testing/debugging
        def check_r(node, parent):
            if node is None:
                return 0
    
            assert node.parent is parent
            assert node.lchild is None or node.priority >= node.lchild.priority
            assert node.rchild is None or node.priority >= node.rchild.priority
            assert node.parent is None or node.priority <= node.parent.priority
            assert 1+check_r(node.lchild, node) + check_r(node.rchild, node) == node.size
            return node.size
    
        assert self.root is None or self.root.parent is None
        return check_r(self.root, None)


The next cell is just to test the implementation of treap.  We take a list of length 20, pick a random permutation of $\{1,\ldots,20\}$ and insert into the treap the elements of the list in order as prescribed by the permutation.  Regardless of the order in which we insert them into the treap, at the end we should get back (via our in-order traversal) the list we started with.

In [13]:
for _ in range(10000):
    my_list = [(0,3), (1,1), (2,2), (3,3), (0,2), (1,3), (2,1), (0,1), (1,2), (0,3), 
               (8,1), (7,3), (6,2), (5,1), (8,2), (7,1), (6,3), (8,3), (7,2), (8,1)]
    my_treap = Treap()
    inserted = set()
    perm = random.sample(range(1,21), 20)
    #print(perm)
    my_events = [None for _ in range(20)]
    for i in range(20):
        vert, color = my_list[perm[i]-1]
        next_event = None
        for j in range(19, perm[i]-1, -1):
            #print(my_events[j])
            if my_events[j] is not None:
                next_event = my_events[j]
        #print(f"perm[i] is {perm[i]} and next_event is {next_event}.")
        temp = Event(vert, color, random.random())
        my_treap.insert_event(temp, next_event)
        inserted |= {perm[i]}
        easy_rank = sum(1 for j in inserted if j <= perm[i])
        assert temp.find_rank() == easy_rank, "Ranks don't match!"
        
        my_events[perm[i]-1] = temp
        assert my_treap.check_invariants() == len(inserted)
        
    #my_treap.inorder_traversal(priority=True, rank=True)
    rseq = my_treap.get_list()
    assert rseq == my_list

Now we come to the actual algorithm ```find_rseq_treap``` to construct a recoloring sequence.  Mainly, this is a wrapper for the function ```extend_rseq_treap```, which does all of the work.  We don't pass much from the wrapper function, just the vertex and its list; most of the data we need is just inherited from the wrapper.  At the end, we verify that our recoloring sequence is indeed valid, before returning it.

In [21]:
def find_rseq_treap(G, coloring1: dict, coloring2: dict, lists: dict, to_add_order: list, info=False):
    coloring_events = {v:[] for v in G}
    recolor_seq = Treap()
    added_verts = set()

    def extend_rseq_treap(v: int, v_colors: set, DEBUG=False):
        nonlocal G, recolor_seq, coloring1, coloring2, coloring_events
        def dprint(a):
            if DEBUG:
                print(a)
    
        v_source, v_target = coloring1[v], coloring2[v]
        nbrs_recolor_seq = []
        v_nbrs = set(G.neighbors(v)) & added_verts
        for w in v_nbrs:
            w_events = [(event.find_rank(), event.vertex, event.color, event) for event in coloring_events[w]]
            nbrs_recolor_seq += w_events
        nbrs_recolor_seq.sort()
        curr_color = {w:coloring1[w] for w in (v_nbrs | {v}) }  # dictionary that maintains colors on N[v]
        how_many = len(v_colors) - len(v_nbrs) - 1
    
        ext_to_v_seq = []
        for i in range(len(nbrs_recolor_seq)):
            num, vert, color, event = nbrs_recolor_seq[i]
            if color != curr_color[v]:
                curr_color[vert] = color
            else:
                to_avoid1 = {curr_color[w] for w in v_nbrs}
                to_avoid2 = {nbrs_recolor_seq[i+j][2] for j in range(min(how_many,len(nbrs_recolor_seq)-i))}
                allowed_colors = v_colors - (to_avoid1 | to_avoid2)
                picked_color = next(iter(allowed_colors))
                curr_color[v] = picked_color
                curr_color[vert] = color
                ext_to_v_seq.append((event, picked_color))
    
        # ext_to_v_seq ... (event, color)  # list: each step recolors v with color, just before event in recolor_seq 
        if (len(ext_to_v_seq) == 0 and v_target != v_source) or (len(ext_to_v_seq) >= 1 and ext_to_v_seq[-1][1] != v_target):
            ext_to_v_seq.append((None, v_target))
        
        for next_event, color in ext_to_v_seq:
            priority = random.random()
            new_event = Event(v,color,priority)
            recolor_seq.insert_event(new_event, next_event) 
            coloring_events[v] += [new_event]
    
    for v in to_add_order:
        extend_rseq_treap(v, lists[v], set(G.neighbors(v))&added_verts)
        added_verts |= {v}
    #print(f"rseq is {rseq}.")
    rseq_as_list = recolor_seq.get_list()
    verify_rseq(G, rseq_as_list, coloring1, coloring2, lists, info=info)
    return rseq_as_list

To test ```find_rseq_treap```, we write a few helper functions, ```build_d_degen```, ```make_lists```, and ```build_random_coloring```. These build random $d$-degenerate graphs (given $d$ and a desired order $n$), random lists (given a graph, a list size, and a pallete size), and a random coloring (given a graph and its lists).

In [16]:
def build_d_degen(n: int, d=1):
    G = nx.complete_graph(d+1)
    G.add_nodes_from(range(d+1,n))
    for v in range(d+1,n):
        nbrhd = random.sample(range(v),d)
        for w in nbrhd:
            G.add_edge(v,w)
    return G

In [17]:
def make_lists(vert_set, union_size, list_size) -> dict:
    my_lists = {}
    for v in vert_set:
        my_lists[v] = set(random.sample(range(union_size), list_size))
    return my_lists

In [18]:
def build_random_coloring(G, lists) -> dict:
    my_colors = {}
    for v in range(len(G)):
        early_nbrs = {w for w in G.neighbors(v) if w < v}
        to_avoid = {my_colors[w] for w in early_nbrs}
        v_color = random.sample(list(lists[v]-to_avoid),1)[0]
        my_colors[v] = v_color
    return my_colors

The main takeaway is that ```find_rseq_treap``` is *fast*.  For a 10,000-vertex 5-degenerate graph, it typically is well under a second.

In [29]:
n = 10000
d = 5
G_random = build_d_degen(n,d)
G_random_lists = make_lists(G_random,2*d, 2*d) #2*(d+1),2*(d+1))
coloring1 = build_random_coloring(G_random, G_random_lists)
coloring2 = build_random_coloring(G_random, G_random_lists)

In [30]:
G_random_rseq = find_rseq_treap(G_random, coloring1, coloring2, G_random_lists, G_random, info=True)

Recoloring sequence is valid.


In [31]:
verify_rseq(G_random, G_random_rseq, coloring1, coloring2, G_random_lists, info=True)

Recoloring sequence is valid.


{'max_times': 4, 'min_times': 0, 'total_times': 14531}

And even for bigger and somewhat denser graphs, time is really not an issue.

In [32]:
n = 100000
d = 15
G_random = build_d_degen(n,d)
G_random_lists = make_lists(G_random,2*d, 2*d) #2*(d+1),2*(d+1))
coloring1 = build_random_coloring(G_random, G_random_lists)
coloring2 = build_random_coloring(G_random, G_random_lists)

In [37]:
G_random_rseq = find_rseq_treap(G_random, coloring1, coloring2, G_random_lists, G_random)
verify_rseq(G_random, G_random_rseq, coloring1, coloring2, G_random_lists, info=True)

Recoloring sequence is valid.
Recoloring sequence is valid.


{'max_times': 3, 'min_times': 0, 'total_times': 148797}

In [177]:
# ToDo list
# write verify_rseq : DONE
# test verify_rseq on path (example that I wrote) : DONE
# write extend_rseq : DONE
# test extend_rseq on paths: DONE
# write build_lists : DONE
# recursively package extend_rseq to get find_rseq : DONE
# build general d-degenerate graphs : DONE
# test on general d-degenerate graphs : somewhat SLOW =(
# build "box" graphs
# test on "box" graphs

In [47]:
path9 = nx.path_graph(9)

In [48]:
path9.edges()

EdgeView([(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 7), (7, 8)])

In [49]:
path9_coloring1 = {i:(i%3)+1 for i in range(9)}

In [50]:
path9_coloring1

{0: 1, 1: 2, 2: 3, 3: 1, 4: 2, 5: 3, 6: 1, 7: 2, 8: 3}

In [51]:
path9_coloring2 = {i:3- i%3 for i in range(9)}

In [52]:
path9_coloring2

{0: 3, 1: 2, 2: 1, 3: 3, 4: 2, 5: 1, 6: 3, 7: 2, 8: 1}

In [61]:
path9_rseq = [(0,3), (1,1), (2,2), (3,3), (0,2), (1,3), (2,1), (0,1), (1,2), (0,3), (8,1), (7,3), (6,2), (5,1), (8,2), (7,1), (6,3), (8,3), (7,2), (8,1)]

In [62]:
path9_lists = {v: {1,2,3} for v in path9}

In [72]:
verify_rseq(path9, path9_rseq, path9_coloring1, path9_coloring2, path9_lists, info=True)

Recoloring sequence is valid.


{'max_times': 4, 'min_times': 0, 'total_times': 20}